# PREPROCESSING

In [2]:
import pandas as pd
import numpy as np
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoNormal
import pyro.optim as optim


Load + split

In [3]:
df = pd.read_csv("data/Cleaned_dataset.csv")

# select the features we want to use for the model (same list)
df = df[[
    "Journey_day", "Airline", "Source", "Departure", "Total_stops",
    "Arrival", "Destination", "Duration_in_hours", "Days_left", "Fare"
]].dropna().copy()

# convert fare currency from INR to USD (keep float; model will standardize anyway)
df["Fare"] = (df["Fare"].astype(float) * 0.01066)

# split into training and test set (keep their shuffle+split structure)
train_fraction = 0.8
df_shuffled = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
split_idx = int(len(df_shuffled) * train_fraction)

df_train_raw = df_shuffled.iloc[:split_idx].copy()
df_test_raw = df_shuffled.iloc[split_idx:].copy()

Encode + scale

In [4]:
def encode_train_test(train_s: pd.Series, test_s: pd.Series):
    train_s = train_s.astype(str)
    test_s = test_s.astype(str)

    cats = sorted(train_s.unique().tolist())
    mapping = {cat: i for i, cat in enumerate(cats)}
    unk_idx = len(mapping)

    train_enc = train_s.map(mapping).fillna(unk_idx).astype(int).to_numpy()
    test_enc = test_s.map(mapping).fillna(unk_idx).astype(int).to_numpy()

    cardinality = unk_idx + 1  # include UNK bucket
    return train_enc, test_enc, cardinality, mapping

# helper: z-score using TRAIN stats only (robust if std=0)
def zscore_train_test(train_col: pd.Series, test_col: pd.Series):
    train_col = train_col.astype(float)
    test_col = test_col.astype(float)

    mean = float(train_col.mean())
    std = float(train_col.std())
    if std == 0 or np.isnan(std):
        std = 1.0

    train_scaled = ((train_col - mean) / std).to_numpy(dtype=np.float32)
    test_scaled = ((test_col - mean) / std).to_numpy(dtype=np.float32)
    return train_scaled, test_scaled, mean, std

# ----- categorical encoding (AFTER split) -----
cat_cols = ["Journey_day", "Airline", "Source", "Departure", "Total_stops", "Arrival", "Destination"]

cat_train = {}
cat_test = {}
cardinalities = []
category_mappings = {}

for col in cat_cols:
    tr, te, card, mapping = encode_train_test(df_train_raw[col], df_test_raw[col])
    cat_train[col] = tr
    cat_test[col] = te
    cardinalities.append(card)
    category_mappings[col] = mapping

# ----- numerical scaling (using TRAIN only) -----
duration_train, duration_test, duration_mean, duration_std = zscore_train_test(
    df_train_raw["Duration_in_hours"], df_test_raw["Duration_in_hours"]
)
days_left_train, days_left_test, days_left_mean, days_left_std = zscore_train_test(
    df_train_raw["Days_left"], df_test_raw["Days_left"]
)

# ----- target scaling (using TRAIN only) -----
fare_train, fare_test, fare_mean, fare_std = zscore_train_test(
    df_train_raw["Fare"], df_test_raw["Fare"]
)



Tensors + summary

In [5]:
# create torch tensors for each categorical variable for both training and test set (same style)
journey_day_tensor_train = torch.tensor(cat_train["Journey_day"], dtype=torch.long)
journey_day_tensor_test = torch.tensor(cat_test["Journey_day"], dtype=torch.long)

airline_tensor_train = torch.tensor(cat_train["Airline"], dtype=torch.long)
airline_tensor_test = torch.tensor(cat_test["Airline"], dtype=torch.long)

source_tensor_train = torch.tensor(cat_train["Source"], dtype=torch.long)
source_tensor_test = torch.tensor(cat_test["Source"], dtype=torch.long)

departure_tensor_train = torch.tensor(cat_train["Departure"], dtype=torch.long)
departure_tensor_test = torch.tensor(cat_test["Departure"], dtype=torch.long)

total_stops_tensor_train = torch.tensor(cat_train["Total_stops"], dtype=torch.long)
total_stops_tensor_test = torch.tensor(cat_test["Total_stops"], dtype=torch.long)

arrival_tensor_train = torch.tensor(cat_train["Arrival"], dtype=torch.long)
arrival_tensor_test = torch.tensor(cat_test["Arrival"], dtype=torch.long)

destination_tensor_train = torch.tensor(cat_train["Destination"], dtype=torch.long)
destination_tensor_test = torch.tensor(cat_test["Destination"], dtype=torch.long)

# create torch tensors for each numerical variable for both training and test set (same style)
duration_in_hours_tensor_train = torch.tensor(duration_train, dtype=torch.float32)
duration_in_hours_tensor_test = torch.tensor(duration_test, dtype=torch.float32)

days_left_tensor_train = torch.tensor(days_left_train, dtype=torch.float32)
days_left_tensor_test = torch.tensor(days_left_test, dtype=torch.float32)

# create torch tensor for target variable for both training and test set (same style)
fare_tensor_train = torch.tensor(fare_train, dtype=torch.float32)
fare_tensor_test = torch.tensor(fare_test, dtype=torch.float32)

# lists of each type of feature (categorical and numerical) (same style)
cat_tensors_train = [
    journey_day_tensor_train, airline_tensor_train, source_tensor_train,
    departure_tensor_train, total_stops_tensor_train, arrival_tensor_train,
    destination_tensor_train
]
cat_tensors_test = [
    journey_day_tensor_test, airline_tensor_test, source_tensor_test,
    departure_tensor_test, total_stops_tensor_test, arrival_tensor_test,
    destination_tensor_test
]
num_tensors_train = [duration_in_hours_tensor_train, days_left_tensor_train]
num_tensors_test = [duration_in_hours_tensor_test, days_left_tensor_test]

print("Cardinalities (incl. UNK):", cardinalities)
print("Fare mean/std (USD):", fare_mean, fare_std)
print("Train size:", len(fare_tensor_train), "Test size:", len(fare_tensor_test))

Cardinalities (incl. UNK): [8, 10, 8, 5, 4, 5, 8]
Fare mean/std (USD): 243.60542306301323 216.63490024301598
Train size: 361670 Test size: 90418
